# Calibration Pipeline Internals

Internal reference for debugging and development of the calibration pipeline.

**Requirements:** `policy_data.db`, `block_cd_distributions.csv.gz`, and the source-imputed stratified CPS H5 file in `STORAGE_FOLDER`.

---
# Part 1: Anatomy of the calibration matrix 

This section demonstrates the structure of the calibration matrix so you can inspect it. We build toward a crucial idea: understand your matrix and the values constraining your problem before you optimize for it.

We build the full calibration matrix using `UnifiedMatrixBuilder` with clone-based geography from `assign_random_geography`, then inspect its structure: what rows and columns represent, how target groups partition the loss function, and where sparsity patterns emerge.

**Column layout:** `col = clone_idx * n_records + record_idx`

## 1.1 Setup

In [3]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
from dataclasses import dataclass
from policyengine_us_data.calibration.calibration_utils import (
    create_target_groups,
    drop_target_groups,
    get_geo_level,
    STATE_CODES,
)

# Toy parameters — used in place of a real CPS H5 to avoid the multi-minute
# runtime of build_matrix() (runs PolicyEngine per state, per clone).
N_CLONES = 3
n_records = 8

In [4]:
# Geography: each of the N_CLONES * n_records columns gets a state + CD.
# The real assign_random_geography() enforces the no-CD-collision invariant;
# here we hand-assign a small example satisfying the same property.

np.random.seed(42)

# state_fips and cd_geoid per column (clone * n_records + record)
_state_per_record = np.array([6, 48, 36, 6, 48, 17, 36, 6])
_cd_per_record = np.array([601, 4801, 3601, 602, 4802, 1701, 3602, 603])


@dataclass
class ToyGeography:
    state_fips: np.ndarray
    cd_geoid: np.ndarray
    block_geoid: np.ndarray


# Replicate across clones with different block assignments
_sf = np.tile(_state_per_record, N_CLONES)
_cd = np.tile(_cd_per_record, N_CLONES)
_bk = np.array([f"{_cd[i]}B{i:04d}" for i in range(len(_sf))])

geography = ToyGeography(
    state_fips=_sf,
    cd_geoid=np.array([str(c) for c in _cd]),
    block_geoid=_bk,
)

# Toy targets.
# geographic_id: 'US' = national, str(FIPS < 100) = state, str(geoid >= 100) = district.
_rows = [
    # domain_variable        variable               geographic_id     value
    ("snap", "snap", "US", 15_000_000_000),
    ("snap", "snap", "6", 5_000_000_000),
    ("snap", "snap", "48", 3_000_000_000),
    ("snap", "snap", "36", 2_000_000_000),
    ("snap", "snap", "17", 1_000_000_000),
    ("snap", "snap_household_count", "US", 5_000_000),
    ("snap", "snap_household_count", "6", 1_500_000),
    ("aca_ptc", "aca_ptc", "US", 8_000_000_000),
    ("aca_ptc", "aca_ptc", "6", 3_000_000_000),
    ("aca_ptc", "aca_ptc", "601", 400_000_000),
    ("employment_income", "employment_income", "US", 9_000_000_000_000),
    ("employment_income", "employment_income", "6", 3_000_000_000_000),
]
targets_df = pd.DataFrame(
    _rows, columns=["domain_variable", "variable", "geographic_id", "value"]
)
targets_df["uprating_factor"] = 1.02
target_names = [
    f"{r.domain_variable}_{r.geographic_id}" for _, r in targets_df.iterrows()
]
n_targets = len(targets_df)

# Build toy sparse matrix.
# X[i, j] = variable value for column j, or 0 if geography doesn't match target i.
rng = np.random.default_rng(0)
_sr, _sc, _sv = [], [], []
for t_idx, row in targets_df.iterrows():
    geo = row["geographic_id"]
    for col in range(N_CLONES * n_records):
        state = str(geography.state_fips[col])
        cd = geography.cd_geoid[col]
        if geo == "US" or geo == state or geo == cd:
            _sr.append(t_idx)
            _sc.append(col)
            _sv.append(rng.uniform(0.5, 5.0))

X_sparse = sp.csr_matrix((_sv, (_sr, _sc)), shape=(n_targets, N_CLONES * n_records))

print(f"Records: {n_records},  Clones: {N_CLONES},  Columns: {N_CLONES * n_records}")
print(f"Matrix shape: {X_sparse.shape}")
print(f"Non-zeros: {X_sparse.nnz}")

Records: 8,  Clones: 3,  Columns: 24
Matrix shape: (12, 24)
Non-zeros: 150


## 1.2 Matrix overview

In [5]:
print(f"Targets: {X_sparse.shape[0]}")
print(f"Columns: {X_sparse.shape[1]:,} ({N_CLONES} clones x {n_records:,} records)")
print(f"Non-zeros: {X_sparse.nnz:,}")
print(f"Density: {X_sparse.nnz / (X_sparse.shape[0] * X_sparse.shape[1]):.6f}")

geo_levels = targets_df["geographic_id"].apply(get_geo_level)
level_names = {0: "National", 1: "State", 2: "District"}
for level in [0, 1, 2]:
    n = (geo_levels == level).sum()
    if n > 0:
        print(f"  {level_names[level]}: {n} targets")

Targets: 12
Columns: 24 (3 clones x 8 records)
Non-zeros: 150
Density: 0.520833
  National: 4 targets
  State: 7 targets
  District: 1 targets


## 1.3 Anatomy of a row

Each row is one calibration target — a known aggregate (dollar total, household count, person count) that the optimizer tries to match. The row vector's non-zero entries identify which cloned records can contribute to that target.

In [6]:
mid_row = X_sparse.shape[0] // 2
row = targets_df.iloc[mid_row]
print(f"Row {mid_row}: {target_names[mid_row]}")
print(f"  variable: {row['variable']}")
print(f"  geographic_id: {row['geographic_id']}")
print(f"  geo_level: {get_geo_level(row['geographic_id'])}")
print(f"  target value: {row['value']:,.0f}")
print(f"  uprating_factor: {row.get('uprating_factor', 'N/A')}")

Row 6: snap_6
  variable: snap_household_count
  geographic_id: 6
  geo_level: 1
  target value: 1,500,000
  uprating_factor: 1.02


In [7]:
row_vec = X_sparse[mid_row, :]
nz_cols = row_vec.nonzero()[1]
print(f"Row {mid_row} has {len(nz_cols):,} non-zero columns")

if len(nz_cols) > 0:
    clone_indices = nz_cols // n_records
    record_indices = nz_cols % n_records
    print(f"  Spans {len(np.unique(clone_indices))} clone(s)")
    print(f"  Spans {len(np.unique(record_indices))} unique record(s)")

    first_col = nz_cols[0]
    print(f"\nFirst non-zero column ({first_col}):")
    print(f"  clone_idx: {first_col // n_records}")
    print(f"  record_idx: {first_col % n_records}")
    print(f"  state_fips: {geography.state_fips[first_col]}")
    print(f"  cd_geoid: {geography.cd_geoid[first_col]}")
    print(f"  value: {X_sparse[mid_row, first_col]:.2f}")

Row 6 has 9 non-zero columns
  Spans 3 clone(s)
  Spans 3 unique record(s)

First non-zero column (0):
  clone_idx: 0
  record_idx: 0
  state_fips: 6
  cd_geoid: 601
  value: 2.48


## 1.4 Anatomy of a column

Each column represents one (record, clone) pair. Columns are organized in clone blocks: the first `n_records` columns belong to clone 0, the next to clone 1, and so on. The block formula is:

$$\text{column\_idx} = \text{clone\_idx} \times n_{\text{records}} + \text{record\_idx}$$

In [8]:
col_idx = 1 * n_records + 3  # clone 1, record 3
clone_idx = col_idx // n_records
record_idx = col_idx % n_records
print(f"Column {col_idx}:")
print(f"  clone_idx: {clone_idx}")
print(f"  record_idx: {record_idx}")
print(f"  state_fips: {geography.state_fips[col_idx]}")
print(f"  cd_geoid: {geography.cd_geoid[col_idx]}")
print(f"  block_geoid: {geography.block_geoid[col_idx]}")

col_vec = X_sparse[:, col_idx]
nz_rows = col_vec.nonzero()[0]
print(f"\nThis column has non-zero values in {len(nz_rows)} target rows")
if len(nz_rows) > 0:
    print("First 5 target rows:")
    for r in nz_rows[:5]:
        row = targets_df.iloc[r]
        print(
            f"  row {r}: {row['variable']} "
            f"(geo={row['geographic_id']}, "
            f"val={X_sparse[r, col_idx]:.2f})"
        )

Column 11:
  clone_idx: 1
  record_idx: 3
  state_fips: 6
  cd_geoid: 602
  block_geoid: 602B0011

This column has non-zero values in 8 target rows
First 5 target rows:
  row 0: snap (geo=US, val=0.51)
  row 1: snap (geo=6, val=3.58)
  row 5: snap_household_count (geo=US, val=0.73)
  row 6: snap_household_count (geo=6, val=3.29)
  row 7: aca_ptc (geo=US, val=0.57)


## 1.5 Target groups and loss weighting

Target groups partition the rows by (domain, variable, geographic level). This grouping can be used to make each group contribute equally to the loss function, so hundreds of district-level rows don't drown out a single national row. However, this is currently not part of the pipeline. In its current form, all targets contribute equally to the loss function.

In [9]:
target_groups, group_info = create_target_groups(targets_df)

records = []
for gid, info in enumerate(group_info):
    mask = target_groups == gid
    vals = targets_df.loc[mask, "value"]
    records.append(
        {
            "group_id": gid,
            "description": info,
            "n_targets": mask.sum(),
            "min_value": vals.min(),
            "median_value": vals.median(),
            "max_value": vals.max(),
        }
    )

group_df = pd.DataFrame(records)
print(group_df.to_string(index=False))


=== Creating Target Groups ===

National targets:
  Group 0: Aca Ptc = 8,000,000,000
  Group 1: Employment Income = 9,000,000,000,000
  Group 2: Snap = 15,000,000,000
  Group 3: SNAP Snap Household Count = 5,000,000

State targets:
  Group 4: Aca Ptc = 3,000,000,000
  Group 5: Employment Income = 3,000,000,000,000
  Group 6: Snap (4 targets)
  Group 7: SNAP Snap Household Count = 1,500,000

District targets:
  Group 8: Aca Ptc = 400,000,000

Total groups created: 9
 group_id                                                             description  n_targets     min_value  median_value     max_value
        0               Group 0: National Aca Ptc (1 target, value=8,000,000,000)          1    8000000000  8.000000e+09    8000000000
        1 Group 1: National Employment Income (1 target, value=9,000,000,000,000)          1 9000000000000  9.000000e+12 9000000000000
        2                 Group 2: National Snap (1 target, value=15,000,000,000)          1   15000000000  1.500000e+10   1

## 1.6 Tracing a household across clones

One CPS record appears once per clone (N_CLONES column positions). Each clone places it in a different census block/CD/state, so it contributes to different geographic targets depending on the clone.

In [10]:
# Trace one SNAP-receiving household across all clones.
# In production this uses sim.calculate('snap', ...); here we find a record
# that contributes to at least one SNAP row in the toy matrix.

snap_mask = targets_df["domain_variable"] == "snap"
snap_rows = np.where(snap_mask)[0]

# Find the first record (base index) with non-zero SNAP contributions
snap_sub = X_sparse[snap_rows, :]  # sub-matrix: SNAP rows only
col_nnz = np.diff(snap_sub.tocsc().indptr)  # nnz per column
example_hh_idx = int(np.where(col_nnz > 0)[0][0]) % n_records

print(f"Example SNAP-contributing household: base record index {example_hh_idx}")

clone_cols = [c * n_records + example_hh_idx for c in range(N_CLONES)]
print(f"\nColumn positions across {N_CLONES} clones:")
for col in clone_cols:
    state = geography.state_fips[col]
    cd = geography.cd_geoid[col]
    nnz = X_sparse[:, col].nnz
    abbr = STATE_CODES.get(state, "??")
    print(f"  col {col}: {abbr} (state={state}, CD={cd}) — {nnz} non-zero rows")

Example SNAP-contributing household: base record index 0

Column positions across 3 clones:
  col 0: CA (state=6, CD=601) — 9 non-zero rows
  col 8: CA (state=6, CD=601) — 9 non-zero rows
  col 16: CA (state=6, CD=601) — 9 non-zero rows


## 1.7 Sparsity analysis

In [11]:
total_cells = X_sparse.shape[0] * X_sparse.shape[1]
density = X_sparse.nnz / total_cells
print(f"Total cells: {total_cells:,}")
print(f"Non-zero entries: {X_sparse.nnz:,}")
print(f"Density: {density:.6f}")
print(f"Sparsity: {1 - density:.4%}")

nnz_per_row = np.diff(X_sparse.indptr)
print(f"\nNon-zeros per row:")
print(f"  min:    {nnz_per_row.min():,}")
print(f"  median: {int(np.median(nnz_per_row)):,}")
print(f"  mean:   {nnz_per_row.mean():,.0f}")
print(f"  max:    {nnz_per_row.max():,}")

geo_levels = targets_df["geographic_id"].apply(get_geo_level)
level_names = {0: "National", 1: "State", 2: "District"}
print("\nBy geographic level:")
for level in [0, 1, 2]:
    mask = (geo_levels == level).values
    if mask.any():
        vals = nnz_per_row[mask]
        print(
            f"  {level_names[level]:10s}: "
            f"n={mask.sum():>4d}, "
            f"median nnz={int(np.median(vals)):>7,}, "
            f"range=[{vals.min():,}, {vals.max():,}]"
        )

Total cells: 288
Non-zero entries: 150
Density: 0.520833
Sparsity: 47.9167%

Non-zeros per row:
  min:    3
  median: 9
  mean:   12
  max:    24

By geographic level:
  National  : n=   4, median nnz=     24, range=[24, 24]
  State     : n=   7, median nnz=      9, range=[3, 9]
  District  : n=   1, median nnz=      3, range=[3, 3]


## 1.8 Dropping targets: groups vs domain filtering

There are two mechanisms for excluding targets from calibration. They operate at different stages and serve different purposes.

### Group-based filtering (not used in production)

`drop_target_groups()` removes entire groups created by `create_target_groups()` from the matrix after it has been built. This was designed for cases where a group is redundant after hierarchical uprating — for example, state-level SNAP Household Count becomes redundant once district-level targets were reconciled to sum to the state totals.

This mechanism is **not currently used in the pipeline** (`target_groups=None` is passed to the optimizer). It is preserved for potential future use with group-weighted loss balancing. The example below demonstrates how it works.

### Domain filtering via `target_config.yaml` (used in production)

In production, target filtering happens **before the matrix is built**, not after. `target_config.yaml` is the authoritative gating list: any database target not matching an `include` entry is discarded by the matrix builder at query time (see section 4.4). This is how targets are added or removed from calibration — by editing the YAML config, not by dropping groups from an already-built matrix.

The key difference: group-based filtering operates on the built matrix (post-hoc removal of rows), while `target_config.yaml` filtering prevents unwanted targets from entering the matrix in the first place.

### Achievable targets

Regardless of which filtering approach is used, a target is achievable only if at least one household can contribute to it (row sum > 0). Rows with sum = 0 are impossible constraints that the optimizer cannot satisfy and are removed before fitting.

In [12]:
GROUPS_TO_DROP = [
    ("SNAP Household Count", "State"),
]

targets_filtered, X_filtered = drop_target_groups(
    targets_df, X_sparse, target_groups, group_info, GROUPS_TO_DROP
)

row_sums = np.array(X_filtered.sum(axis=1)).flatten()
achievable_mask = row_sums > 0
n_achievable = achievable_mask.sum()
n_impossible = (~achievable_mask).sum()

print(f"\nAchievable targets: {n_achievable}")
print(f"Impossible targets: {n_impossible}")

X_final = X_filtered[achievable_mask, :]
print(f"\nFinal matrix shape: {X_final.shape}")
print(f"Final non-zero entries: {X_final.nnz:,}")
print(f"This is what the optimizer receives.")

Matrix before: 12 rows
  DROPPING Group 7: State SNAP Snap Household Count (1 target, value=1,500,000) (1 rows)

  KEEPING  Group 0: National Aca Ptc (1 target, value=8,000,000,000) (1 rows)
  KEEPING  Group 1: National Employment Income (1 target, value=9,000,000,000,000) (1 rows)
  KEEPING  Group 2: National Snap (1 target, value=15,000,000,000) (1 rows)
  KEEPING  Group 3: National SNAP Snap Household Count (1 target, value=5,000,000) (1 rows)
  KEEPING  Group 4: State Aca Ptc (1 target, value=3,000,000,000) (1 rows)
  KEEPING  Group 5: State Employment Income (1 target, value=3,000,000,000,000) (1 rows)
  KEEPING  Group 6: State Snap (4 targets) (4 rows)
  KEEPING  Group 8: District Aca Ptc (1 target, value=400,000,000) (1 rows)

Matrix after: 11 rows

Achievable targets: 11
Impossible targets: 0

Final matrix shape: (11, 24)
Final non-zero entries: 141
This is what the optimizer receives.


---
# Part 2: Calibration matrix assembly — per-state simulation

The previous section showed *what* the matrix contains. This section explains *how* `UnifiedMatrixBuilder` fills it in — specifically the per-state simulation pass, domain constraints that gate matrix rows, and the special treatment of county-dependent variables.

Because the matrix builder runs inside worker processes and cannot share live Python objects across process boundaries, all simulation is done with picklable top-level functions rather than methods.

## 2.1 Per-state simulation with parallel workers

Before the clone loop begins, the builder dispatches one simulation job per unique state FIPS that appears across all clone assignments. Each job runs in a `ProcessPoolExecutor` worker and calls `_compute_single_state()` — a module-level function (not a method) so it is picklable.

Inside `_compute_single_state()`, the worker:

1. Creates a fresh `Microsimulation` from the base CPS H5 file.
2. Overwrites every household's `state_fips` with a uniform array set to the target state.
3. Invalidates cached downstream variables with `delete_arrays()` so PolicyEngine recomputes them under the new state.
4. Calculates each target variable mapped to the `household` entity.
5. Calculates each constraint variable mapped to the `person` entity (constraints need person-level resolution).

```python
# From unified_matrix_builder.py — _compute_single_state()
state_sim = Microsimulation(dataset=dataset_path)

state_sim.set_input(
    "state_fips",
    time_period,
    np.full(n_hh, state, dtype=np.int32),
)
for var in get_calculated_variables(state_sim):
    state_sim.delete_arrays(var)

hh = {}
for var in target_vars:
    if var.endswith("_count"):
        continue
    hh[var] = state_sim.calculate(var, time_period, map_to="household").values.astype(np.float32)

person = {}
for var in constraint_vars:
    person[var] = state_sim.calculate(var, time_period, map_to="person").values.astype(np.float32)
```

The return value is a `(state_fips, {"hh": ..., "person": ..., "entity": ..., "entity_wf_false": ...})` tuple. After all state workers finish, the builder collects results into a `state_values` dict keyed by FIPS.

## 2.2 Clone loop: slicing and assembling

With precomputed state values in hand, `_process_single_clone()` runs once per clone (also in worker processes, sharing read-only data via `_init_clone_worker()`). Each call:

1. Reads its slice of the geography arrays: `clone_states = geo_states[col_start:col_end]`.
2. Calls `_assemble_clone_values_standalone()`, which fans out state-level arrays into a per-record array by applying per-state masks:

```python
# From _assemble_clone_values_standalone()
arr = np.empty(n_records, dtype=np.float32)
for state in unique_clone_states:
    mask = state_masks[int(state)]          # records assigned to this state
    arr[mask] = state_values[int(state)]["hh"][var][mask]
hh_vars[var] = arr
```

3. Evaluates domain constraints (§2.4) and writes non-zero COO entries for every target row.

Column layout: `col_idx = clone_idx * n_records + record_idx`, so `col_start = clone_idx * n_records` and `col_end = col_start + n_records`.

## 2.3 Domain constraints: gating which records contribute

Each target row in the matrix can have a *domain constraint* — a predicate that must be true for a record to contribute a non-zero value to that row. A few common examples:

| Target | Constraint variable | Constraint | Meaning |
|---|---|---|---|
| `aca_ptc` (national, IRS) | `aca_ptc` | `> 0` | Only tax units that receive ACA PTC count |
| `refundable_ctc` (national) | `refundable_ctc` | `> 0` | Only households with positive refundable CTC |
| `self_employment_income` (national) | `self_employment_income` | `> 0` | Only households with SE income |

Constraints come from the `stratum_constraints` table in `policy_data.db`. Each target belongs to a stratum, and each stratum can have one or more constraints stored as `(constraint_variable, operation, value)` rows. The ETL scripts (`db/etl_*.py`) populate these constraints when they insert targets. The matrix builder retrieves them via `_get_stratum_constraints(stratum_id)` for each target, separates geographic constraints (like `state_fips` and `congressional_district_geoid`) from non-geographic ones, and passes the non-geo constraints into the clone loop.

During the clone loop, `_evaluate_constraints_standalone()` applies the predicate at person level and aggregates to household level via `.any()`. A record that does *not* satisfy the constraint contributes 0 to that matrix row — even if it has a non-zero value for the target variable. For the IRS filer-count targets, `tax_unit_is_filer` plays a similar role: only filer tax units appear in those rows.

```python
# From _evaluate_constraints_standalone()
person_mask = np.ones(n_persons, dtype=bool)
for c in constraints:
    vals = person_vars[c["variable"]]
    person_mask &= apply_op(vals, c["operation"], c["value"])

# Aggregate: a household satisfies the constraint if any of its members do
df["satisfies"] = person_mask
hh_mask = df.groupby("household_id")["satisfies"].any()
```

The final matrix entry is:

```python
# From _calculate_target_values_standalone()
vals = hh_vars.get(target_variable)
return (vals * mask).astype(np.float32)  # mask = 0 for records failing constraint
```

## 2.4 Takeup re-randomization in the clone loop

Several programs in PolicyEngine require a *take-up* draw: a stochastic binary decision representing whether an eligible household actually participates. The draws must be **reproducible and consistent** across the matrix builder and the H5 builder — if the two builds use different draws, the matrix rows target a different subpopulation than what ends up in the H5, breaking calibration.

### Why takeup lives in the matrix builder

Takeup draws depend on geography — each entity's take-up rate is resolved from the state FIPS of the census block assigned to that clone. Since different clones of the same household land in different states, they can have different take-up rates and therefore different draws. This per-clone variation is what makes takeup a matrix-building concern rather than a dataset-building concern.

### `SIMPLE_TAKEUP_VARS` and `TAKEUP_AFFECTED_TARGETS`

`SIMPLE_TAKEUP_VARS` (in `utils/takeup.py`) is the canonical list of takeup variables. Each entry is a dict with four keys:

| Key | Meaning |
|---|---|
| `variable` | PolicyEngine boolean input variable (e.g., `takes_up_snap_if_eligible`) |
| `entity` | Native entity of the variable (`spm_unit`, `tax_unit`, or `person`) |
| `rate_key` | Key used to look up the take-up rate from the parameters store |
| `target` | Corresponding calibration target variable, or `None` for non-target draws |

Current entries cover: `snap` (spm_unit), `aca_ptc` (tax_unit), `dc_property_tax_credit` (tax_unit), `head_start` (person), `early_head_start` (person), `ssi` (person), `medicaid` (person), `tanf` (spm_unit), and `would_file_taxes_voluntarily` (tax_unit, no target).

`TAKEUP_AFFECTED_TARGETS` is derived automatically from `SIMPLE_TAKEUP_VARS` — it maps each calibration target that has a takeup variable to `{takeup_var, entity, rate_key}`. Before the clone loop begins, the matrix builder matches `TAKEUP_AFFECTED_TARGETS` against the actual target variables to build `affected_target_info`, which tells each clone worker which target rows need takeup blending.

### Step 1: State precomputation (`_compute_single_state`)

When `rerandomize_takeup=True`, each state's precomputation runs PolicyEngine multiple times:

1. **Baseline simulation** — computes household and person values with the dataset's original takeup values. These populate `hh` (for non-takeup-affected targets) and `person` (for constraint evaluation).

2. **All-takeup-true simulation** — sets every `SIMPLE_TAKEUP_VARS` entry to `True`, clears formula caches, then recalculates each takeup-affected target at entity level. Stored in `entity_vals`. This gives the "what would this entity's value be if it participated in every program?" answer.

3. **Would-file-false simulation** (tax_unit targets only) — sets `would_file_taxes_voluntarily = False`, clears caches, recalculates tax_unit targets. Stored in `entity_wf_false`. This gives the alternative value for non-filers.

These precomputed values are shared read-only data for all clone workers.

### Step 2: Clone loop — drawing and blending

Within `_process_single_clone()`, takeup runs in two phases after assembling per-clone values:

**Phase 1 — Non-target draws (line 703):** Draws for variables where `target is None` — currently just `would_file_taxes_voluntarily`. These are computed via `compute_block_takeup_for_entities()` and stored in `wf_draws`, keyed by entity. They must be drawn before Phase 2 because tax_unit targets depend on them.

**Phase 2 — Target value assembly (line 729):** For each takeup-affected target (e.g., `snap`, `aca_ptc`):
1. Retrieves the precomputed `entity_vals[tvar]` (all-takeup-true eligible value) from the state sim.
2. For tax_unit targets: uses `np.where(wf_draws["tax_unit"], entity_vals[tvar], entity_wf_false[tvar])` to select between the all-true value and the would-file-false value, based on the Phase 1 `would_file` draw.
3. Draws a per-entity program takeup boolean via `compute_block_takeup_for_entities()`.
4. Multiplies: `entity_value = eligible_value * takeup_draw`.
5. Aggregates to household level via `np.add.at()` and overwrites `hh_vars[tvar]`.

### Seeding strategy and the correctness invariant

`compute_block_takeup_for_entities()` uses a `(variable_name, household_id, clone_idx)` triple as the RNG seed:

```python
for hh_id in np.unique(entity_hh_ids):
    hh_mask = entity_hh_ids == hh_id
    for ci in np.unique(entity_clone_indices[hh_mask]):
        ci_mask = hh_mask & (entity_clone_indices == ci)
        n_ent = int(ci_mask.sum())
        rng = seeded_rng(var_name, salt=f"{int(hh_id)}:{int(ci)}")
        draws[ci_mask] = rng.random(n_ent)
```

This guarantees:

- **Same `(var_name, hh_id, clone_idx)`** → same RNG seed → same draw, regardless of call order or process.
- **Different households** → different seeds → independent draws.
- **Different clones of the same household** → different seeds → independent assignments across clones.

**Critical correctness invariant:** The matrix builder and the H5 builder (`publish_local_area.py`) call `compute_block_takeup_for_entities()` independently, but they must pass the same `var_name`, the same `entity_hh_ids`, and the same `clone_idx` for each record. If either side passes a different `(hh_id, clone_idx)` combination, the draws diverge, and the matrix rows target a different subpopulation than what ends up in the H5.

### Rate resolution

`_resolve_rate()` handles both scalar and state-keyed rates. State FIPS is derived from the first two characters of the census block GEOID, so each entity's rate reflects the state it was assigned to in that clone.

### Determinism demo

The following cell verifies that `compute_block_takeup_for_entities()` produces identical draws regardless of call order — the invariant that makes the matrix builder and H5 builder consistent. The matrix builder calls it once per clone (passing a single clone's entities), while the H5 builder may call it with all clones at once. Both must produce the same draws.

In [13]:
import numpy as np
from policyengine_us_data.utils.takeup import compute_block_takeup_for_entities

# Fake household IDs and clone indices
# 4 households, 2 clones each -> 8 (hh, clone) pairs
# Each household has 1 spm_unit for simplicity
hh_ids = np.array([101, 101, 202, 202, 303, 303, 404, 404], dtype=np.int64)
clone_idxs = np.array([0, 1, 0, 1, 0, 1, 0, 1], dtype=np.int64)
# Census block GEOIDs — first two digits = state FIPS
# Using state 06 (CA) for all, so rate is resolved from state code 'CA'
blocks = np.array(["060750001001001"] * 8)

# Scalar take-up rate of 0.75
rate = 0.75
var = "takes_up_snap_if_eligible"

# First call: all 8 entities together (as H5 builder would)
draws_full = compute_block_takeup_for_entities(var, rate, blocks, hh_ids, clone_idxs)

# Second call: split by clone, then concatenate (as matrix builder would)
mask_c0 = clone_idxs == 0
mask_c1 = clone_idxs == 1

draws_c0 = compute_block_takeup_for_entities(
    var, rate, blocks[mask_c0], hh_ids[mask_c0], clone_idxs[mask_c0]
)
draws_c1 = compute_block_takeup_for_entities(
    var, rate, blocks[mask_c1], hh_ids[mask_c1], clone_idxs[mask_c1]
)

# Reconstruct in original order
draws_split = np.empty(8, dtype=bool)
draws_split[mask_c0] = draws_c0
draws_split[mask_c1] = draws_c1

print("Full call draws:  ", draws_full.astype(int).tolist())
print("Split call draws: ", draws_split.astype(int).tolist())
print("Match:", np.array_equal(draws_full, draws_split))

# Show that clone 0 and clone 1 of the same household differ
print("\nPer (hh_id, clone_idx):")
for i, (hh, ci, d) in enumerate(zip(hh_ids, clone_idxs, draws_full)):
    print(f"  hh={hh}, clone={ci} -> takeup={int(d)}")

Full call draws:   [1, 1, 1, 0, 1, 1, 0, 0]
Split call draws:  [1, 1, 1, 0, 1, 1, 0, 0]
Match: True

Per (hh_id, clone_idx):
  hh=101, clone=0 -> takeup=1
  hh=101, clone=1 -> takeup=1
  hh=202, clone=0 -> takeup=1
  hh=202, clone=1 -> takeup=0
  hh=303, clone=0 -> takeup=1
  hh=303, clone=1 -> takeup=1
  hh=404, clone=0 -> takeup=0
  hh=404, clone=1 -> takeup=0


## 2.5 County-dependent variables (e.g., `aca_ptc`)

`COUNTY_DEPENDENT_VARS = {"aca_ptc"}` marks variables whose simulated value depends on county-level premium data. ACA PTC eligibility uses county-level benchmark plan premiums, so a household in LA County gets a different premium slot than the same household in Sacramento County, even if the state is the same.

The builder handles this with a separate `_compute_single_state_group_counties()` pass that runs one simulation per county (reusing a single `Microsimulation` instance per state for efficiency):

```python
# From _compute_single_state_group_counties()
state_sim = Microsimulation(dataset=dataset_path)
state_sim.set_input("state_fips", time_period, np.full(n_hh, state_fips, dtype=np.int32))

for county_fips in counties:
    county_idx = get_county_enum_index_from_fips(county_fips)
    state_sim.set_input("county", time_period, np.full(n_hh, county_idx, dtype=np.int32))
    # delete cached downstream arrays (excluding county and zip_code)
    for var in get_calculated_variables(state_sim):
        if var not in ("county", "zip_code"):
            state_sim.delete_arrays(var)

    hh[var] = state_sim.calculate(var, time_period, map_to="household").values
```

During clone assembly, `_assemble_clone_values_standalone()` checks whether each target variable is in `COUNTY_DEPENDENT_VARS`. If it is, the assembler fills records using county-keyed results rather than state-keyed results:

```python
if var in cdv and county_values and clone_counties is not None:
    for county in unique_counties:
        mask = county_masks[county]
        county_hh = county_values.get(county, {}).get("hh", {})
        if var in county_hh:
            arr[mask] = county_hh[var][mask]
        else:
            # Fall back to state-level if this county wasn't simulated
            st = int(county[:2])
            arr[mask] = state_values[st]["hh"][var][mask]
```

## 2.6 COO assembly: from clone entries to sparse matrix

After state precomputation, takeup, and constraint evaluation, each clone worker builds its contribution to the calibration matrix as COO (coordinate) entries — a list of `(row, col, value)` triples. This allows efficiently storing non-zero values without having to store a large number of zero-valued matrix cells.

### Per-clone entry generation

Inside `_process_single_clone()`, the final loop iterates over every target row:

1. **Geographic filtering** — determines which columns (records) are relevant. District targets use `cd_to_cols`, state targets use `state_to_cols`, national targets use all columns. Only columns belonging to the current clone slice (`col_start..col_end`) are retained.

2. **Value computation** — for count targets (`*_count`), calls `_calculate_target_values_standalone()` which counts entities satisfying the constraint. For dollar targets, multiplies the household variable value by the domain constraint mask. Both use caches (`count_cache`, `mask_cache`) keyed by constraint tuple to avoid redundant computation.

3. **Nonzero filtering** — only entries where `value != 0` are emitted. This is where the matrix gets its sparsity: records outside the target's geography, failing the domain constraint, or with zero variable value produce no entry.

```python
# Simplified from _process_single_clone()
for row_idx in range(n_targets):
    # Geographic filter: which columns belong to this target's area?
    if geo_level == "district":
        clone_cols = cd_to_cols[geo_id]  # columns in this CD
    elif geo_level == "state":
        clone_cols = state_to_cols[geo_id]
    else:
        clone_cols = all_columns
    clone_cols = clone_cols[(clone_cols >= col_start) & (clone_cols < col_end)]

    # Value: variable * constraint mask (or entity count)
    values = source_vars[variable] * constraint_mask

    # Only emit nonzero entries
    vals = values[clone_cols - col_start]
    nonzero = vals != 0
    rows_list.append(np.full(nonzero.sum(), row_idx))
    cols_list.append(clone_cols[nonzero])
    vals_list.append(vals[nonzero])
```

Each clone worker writes its COO entries to a compressed `.npz` file (when using parallel workers) or appends to in-memory lists.

### Final assembly

After all clones finish, the builder concatenates every clone's `(rows, cols, vals)` arrays and constructs a single Compressed Sparse Row (CSR) with `scipy.sparse.csr_matrix`:

```python
# From build_matrix(), step 6
for ci in range(n_clones):
    data = np.load(clone_dir / f"clone_{ci:04d}.npz")
    all_r.append(data["rows"])
    all_c.append(data["cols"])
    all_v.append(data["vals"])

X_csr = sparse.csr_matrix(
    (np.concatenate(all_v), (np.concatenate(all_r), np.concatenate(all_c))),
    shape=(n_targets, n_total),
)
```

The resulting matrix has shape `(n_targets, n_records * n_clones)` — rows are calibration targets, columns are cloned household records. This is what the L0 optimizer receives.

---
# Part 3: Configuring calibration targets after matrix build

## 3.1 `target_config.yaml` include/exclude rules

`target_config.yaml` controls which targets from the built matrix are passed to the L0 optimizer. It is applied **after** the matrix is built, not during construction — the full unfiltered calibration package is saved first (Step 6b in `run_calibration`), then `apply_target_config()` filters targets for fitting (Step 6c). This design lets the same expensive matrix package be reused with different configs without rebuilding.

The filtering works by matching rows in `targets_df` against include/exclude rules. If `include` rules are present, only matching targets survive. If `exclude` rules are also present, they remove from the included set. The corresponding rows in `X_sparse` and `target_names` are dropped in sync.

```python
# From run_calibration(), Step 6c
if target_config:
    targets_df, X_sparse, target_names = apply_target_config(
        targets_df, X_sparse, target_names, target_config
    )
```

Each rule in `target_config.yaml` has three fields:

| Field | Required | Meaning |
|---|---|---|
| `variable` | yes | PolicyEngine variable name |
| `geo_level` | yes | One of `district`, `state`, `national` |
| `domain_variable` | no | If present, only matches targets whose stratum has this constraint variable in the DB |

The `domain_variable` field here is a **row-matching filter**, not a constraint definition. It narrows which rows in the already-built matrix are kept for optimization. The actual domain constraints that gate which records contribute to each matrix row are stored in `stratum_constraints` in `policy_data.db` and are applied during matrix construction (see section 2.3).

Example entries from `target_config.yaml`:

```yaml
include:
  # ACA PTC at district level
  - variable: aca_ptc
    geo_level: district

  # ACA PTC at national level — only the domain-constrained target
  - variable: aca_ptc
    geo_level: national
    domain_variable: aca_ptc

  # SNAP at state level
  - variable: snap
    geo_level: state
```

The inline comments in `target_config.yaml` document why specific targets were removed (e.g., `# REMOVED: state_income_tax — ETL hardcodes $0 for WA and NH`).

### Distinction from `target_filter`

The matrix builder's `_query_targets()` accepts a separate `target_filter` dict (populated from `--domain-variables` CLI args) that filters targets at DB query time — *during* the build. In the default pipeline this filter is empty, so all active targets from `policy_data.db` enter the matrix. `target_config.yaml` then selects the subset to optimize against.

## 3.2 Achievable target filtering

After `target_config.yaml` filtering, a target is **achievable** only if at least one record in the matrix can contribute to it — i.e., the row sum is positive. Targets with row sum = 0 are impossible constraints: no combination of weights can make the weighted sum match a nonzero target value if every entry in that row is zero.

This check runs immediately before the optimizer call in `run_calibration()` (Step 7):

```python
row_sums = np.array(X_sparse.sum(axis=1)).flatten()
achievable = row_sums > 0
```

The `achievable` boolean array is passed to `fit_l0_weights()`, which uses it to exclude impossible targets from the loss function. This prevents the optimizer from wasting gradient signal on targets it can never satisfy.

Common causes of unachievable targets:
- A geographic target (e.g., a specific CD) where no clones landed in that area after the no-collision constraint
- A domain-constrained target where the constraint variable is zero for all records (e.g., a program that no CPS respondent participates in)
- A target that was active in the DB but whose variable isn't computed by the current version of PolicyEngine

---
# Part 4: Hierarchical uprating

Calibration targets in `policy_data.db` come from different sources, at different geographic levels, and from different time periods. Before we can use them in calibration, two adjustments are needed:

1. **Uprating factor (UF)**: Bridges the time gap between the source data's period and the calibration year. For most domains, dollar-valued targets use CPI and count targets use population growth. For **ACA PTC**, we use real state-level enrollment and average APTC changes from CMS/KFF data, giving each state its own UF.

2. **Hierarchy inconsistency factor (HIF)**: Corrects for the fact that district-level totals from one source may not sum to the state-level total from another. This is a pure base-year geometry correction with no time dimension.

These two factors are **separable by linearity**. For each congressional district row:

$$\text{value} = \text{original\_value} \times \text{HIF} \times \text{UF}$$

where $\text{HIF} = S_{\text{base}} \;/\; \sum_i CD_{i,\text{base}}$ and the sum constraint holds:

$$\sum_i (CD_i \times \text{HIF} \times \text{UF}) = \text{UF} \times S_{\text{base}} = S_{\text{uprated}}$$

Two example domains:
- **ACA PTC** (IRS data): Districts sum exactly to state totals, so HIF = 1.0 everywhere. The UF varies by state, reflecting real enrollment and APTC changes between 2022 and 2024.
- **SNAP** (USDA data): District household counts substantially undercount the state administrative totals, so HIF > 1 (often 1.2 to 1.7). The SNAP data is already at the target period, so UF = 1.0.

## 4.1 Raw targets and generic uprating

In [ ]:
# ─── Part 4 setup ──────────────────────────────────────────────────────────
# The uprating walkthrough below requires a fully initialized UnifiedMatrixBuilder
# and a Microsimulation. These load the full source-imputed H5 and policy_data.db.
#
# from policyengine_us import Microsimulation
# from policyengine_us_data import EnhancedCPS_2024
# from policyengine_us_data.storage import STORAGE_FOLDER
# from policyengine_us_data.calibration.unified_matrix_builder import UnifiedMatrixBuilder

# sim = Microsimulation(dataset=EnhancedCPS_2024)
# db_path = str(STORAGE_FOLDER / "calibration" / "policy_data.db")
# builder = UnifiedMatrixBuilder(
#     db_uri=f"sqlite:///{db_path}",
#     time_period=2024,
#     dataset_path=str(EnhancedCPS_2024.file_path),
# )
#
# Without the above, Part 4 cells print a skip message and set outputs to None.

try:
    builder
    sim
except NameError:
    builder = None
    sim = None
    print("Part 4: builder/sim not initialized — cells will skip gracefully.")
    print("Uncomment the setup lines above and re-run to execute the full walkthrough.")

In [23]:
DOMAINS = ["aca_ptc", "snap"]

if builder is not None:
    raw = builder._query_targets({"domain_variables": DOMAINS})
    summary = (
        raw.groupby(["domain_variable", "geo_level", "variable", "period"])
        .agg(count=("value", "size"), total_value=("value", "sum"))
        .reset_index()
    )
    display(summary)
else:
    raw = None
    print("Skipping — builder not initialized.")

,domain_variable,geo_level,variable,period,count,total_value
0,aca_ptc,district,aca_ptc,2022,436,1.419185e+10
1,aca_ptc,district,tax_unit_count,2022,436,6.848330e+06
2,aca_ptc,national,aca_ptc,2022,1,1.419185e+10
3,aca_ptc,national,person_count,2024,1,1.974369e+07
4,aca_ptc,national,tax_unit_count,2022,1,6.848330e+06
5,aca_ptc,state,aca_ptc,2022,51,1.419185e+10
6,aca_ptc,state,tax_unit_count,2022,51,6.848330e+06
7,snap,district,household_count,2024,436,1.563268e+07
8,snap,state,household_count,2024,51,2.217709e+07
9,snap,state,snap,2024,51,9.365787e+10


In [24]:
if builder is not None:
    params = sim.tax_benefit_system.parameters
    uprating_factors = builder._calculate_uprating_factors(params)
    for (yr, kind), f in sorted(uprating_factors.items()):
        if f != 1.0:
            print(f"  {yr} -> 2024 ({kind}): {f:.6f}")
else:
    uprating_factors = None
    print("Skipping — builder not initialized.")

  2022 -> 2024 (cpi): 1.101889
  2022 -> 2024 (pop): 1.020415
  2023 -> 2024 (cpi): 1.035512
  2023 -> 2024 (pop): 1.010947
  2025 -> 2024 (cpi): 0.970879
  2025 -> 2024 (pop): 0.990801


## 4.2 Hierarchical reconciliation

For each (state, variable) pair within a domain:

- **HIF** = `state_original / sum(cd_originals)` — pure base-year correction
- **UF** = state-specific uprating factor:
  - For **ACA PTC**: loaded from `aca_ptc_multipliers_2022_2024.csv` (CMS/KFF enrollment data)
  - For other domains: national CPI/pop factors as fallback

In [25]:
if builder is not None and raw is not None:
    raw["original_value"] = raw["value"].copy()
    raw["uprating_factor"] = raw.apply(
        lambda r: builder._get_uprating_info(
            r["variable"], r["period"], uprating_factors
        )[0],
        axis=1,
    )
    raw["value"] = raw["original_value"] * raw["uprating_factor"]
    result = builder._apply_hierarchical_uprating(raw, DOMAINS, uprating_factors)
else:
    result = None
    print("Skipping — builder not initialized.")

In [26]:
sample_states = {6: "CA", 48: "TX", 36: "NY"}


def show_reconciliation(result, raw, domain, sample_states):
    domain_rows = result[result["domain_variable"] == domain]
    cd_domain = domain_rows[domain_rows["geo_level"] == "district"]
    if cd_domain.empty:
        print("  (no district rows)")
        return
    for fips, abbr in sample_states.items():
        cd_state = cd_domain[
            cd_domain["geographic_id"].apply(
                lambda g, s=fips: int(g) // 100 == s if g not in ("US",) else False
            )
        ]
        if cd_state.empty:
            continue
        for var in sorted(cd_state["variable"].unique()):
            var_rows = cd_state[cd_state["variable"] == var]
            hif = var_rows["hif"].iloc[0]
            uf = var_rows["state_uprating_factor"].iloc[0]
            cd_sum = var_rows["value"].sum()
            print(
                f"  {abbr} {var:20s}  "
                f"hif={hif:.6f}  "
                f"uprating={uf:.6f}  "
                f"sum(CDs)={cd_sum:>14,.0f}"
            )


if result is not None:
    print("ACA PTC (HIF=1.0, state-varying UF):")
    show_reconciliation(result, raw, "aca_ptc", sample_states)
    print("\nSNAP (HIF>1, UF=1.0):")
    show_reconciliation(result, raw, "snap", sample_states)
else:
    print("Skipping — result not available.")

ACA PTC (HIF=1.0, state-varying UF):
  CA aca_ptc               hif=1.000000  uprating=1.209499  sum(CDs)= 3,332,007,010
  CA tax_unit_count        hif=1.000000  uprating=1.055438  sum(CDs)=     1,302,653
  TX aca_ptc               hif=1.000000  uprating=1.957664  sum(CDs)= 2,270,594,110
  TX tax_unit_count        hif=1.000000  uprating=1.968621  sum(CDs)=     1,125,834
  NY aca_ptc               hif=1.000000  uprating=1.343861  sum(CDs)= 2,049,797,288
  NY tax_unit_count        hif=1.000000  uprating=1.075089  sum(CDs)=       593,653

SNAP (HIF>1, UF=1.0):
  CA household_count       hif=1.681273  uprating=1.000000  sum(CDs)=     3,128,640
  TX household_count       hif=1.244524  uprating=1.000000  sum(CDs)=     1,466,107
  NY household_count       hif=1.344447  uprating=1.000000  sum(CDs)=     1,707,770


## 4.3 Verification: sum(CDs) == uprated state

The core invariant: for every (state, variable) pair that has district rows, the sum of reconciled district values must equal the uprated state total.

In [27]:
if result is not None and raw is not None:
    all_ok = True
    checks = 0
    for domain in DOMAINS:
        domain_result = result[result["domain_variable"] == domain]
        cd_result = domain_result[domain_result["geo_level"] == "district"]
        if cd_result.empty:
            continue
        for fips, abbr in sorted(STATE_CODES.items()):
            cd_rows = cd_result[
                cd_result["geographic_id"].apply(
                    lambda g, s=fips: int(g) // 100 == s if g not in ("US",) else False
                )
            ]
            if cd_rows.empty:
                continue
            for var in cd_rows["variable"].unique():
                var_rows = cd_rows[cd_rows["variable"] == var]
                cd_sum = var_rows["value"].sum()
                st = raw[
                    (raw["geo_level"] == "state")
                    & (raw["geographic_id"] == str(fips))
                    & (raw["variable"] == var)
                    & (raw["domain_variable"] == domain)
                ]
                if st.empty:
                    continue
                state_original = st["original_value"].iloc[0]
                state_uf = var_rows["state_uprating_factor"].iloc[0]
                expected = state_original * state_uf
                ok = np.isclose(cd_sum, expected, rtol=1e-6)
                checks += 1
                if not ok:
                    print(
                        f"  FAIL [{domain}] {abbr} {var}: "
                        f"sum(CDs)={cd_sum:.2f} != "
                        f"expected={expected:.2f}"
                    )
                    all_ok = False

    print(
        f"  {checks} checks across {len(DOMAINS)} domains: "
        + ("ALL PASSED" if all_ok else "SOME FAILED")
    )
else:
    print("Skipping — result not available.")

  153 checks across 2 domains: ALL PASSED


---
# Part 5: Calibration package and initial weights

## 5.1 Calibration package serialization

Building the calibration matrix is the most expensive step in the pipeline — it requires running PolicyEngine simulations for every state across all clones. To avoid rebuilding when experimenting with different `target_config.yaml` settings or hyperparameters, `run_calibration()` saves the **full unfiltered** matrix as a calibration package (Step 6b) before applying any config filtering.

`save_calibration_package()` serializes:

| Field | Contents |
|---|---|
| `X_sparse` | The full sparse matrix (all targets, all records) |
| `targets_df` | Target metadata DataFrame (variable, value, geo_level, domain, etc.) |
| `target_names` | Human-readable target name strings |
| `initial_weights` | Population-proportional starting weights (see 5.2) |
| `cd_geoid` | Congressional district GEOID per record (from geography assignment) |
| `block_geoid` | Census block GEOID per record |
| `metadata` | Provenance: git SHA, dataset path, checksums, creation timestamp |

The package is a pickle file. `load_calibration_package()` restores it and runs provenance checks:
- Prints the git SHA and dataset path from `metadata`
- Warns if the package is stale (created more than 7 days ago or from a different git SHA than the current checkout)

The `fit_from_package_*` Modal functions in `remote_calibration_runner.py` use this workflow: build the package once with `build_package_remote()`, then call `fit_from_package_*()` repeatedly with different configs/hyperparameters.

## 5.2 Initial weight computation

The L0 optimizer needs a starting weight for each record. Rather than uniform initialization, `compute_initial_weights()` in `unified_calibration.py` uses **age-bin population targets** to set per-CD proportional weights:

1. Find all `person_count` targets where `domain_variable == "age"` and `geo_level == "district"`.
2. For each congressional district, sum the age-bin target values to get the district's total population.
3. Identify which matrix columns (records) have nonzero entries in those target rows — these are the records active in that CD.
4. Set each record's initial weight to `district_population / n_active_records`.

```python
# From compute_initial_weights()
for cd_id, group in cd_groups:
    cd_pop = group["value"].sum()
    # Find columns with nonzero entries in this CD's age rows
    col_set = set()
    for ri in group.index:
        col_set.update(X_sparse[ri].indices)
    w = cd_pop / len(col_set)
    for c in col_set:
        initial_weights[c] = w
```

This gives the optimizer a head start: records in high-population districts begin with higher weights, and records in small districts begin with lower weights. Without this, the optimizer would spend early epochs just learning the population scale of each district.

If no age-bin district targets are found (e.g., when running with a minimal config), the function falls back to uniform weights of 100.

---

After Part 5, the matrix, filtered targets, initial weights, and achievable mask are ready. See `optimization_and_local_dataset_assembly_internals.ipynb` for how the L0 optimizer uses these inputs to find calibrated weights and how the final H5 datasets are assembled.